# Medidas de Posição

Este notebook tem como objetivo aplicar as **medidas de posição** sobre os dados de desempenho das equipes da NBA, com foco na comparação entre times jogando em casa e fora de casa.

As medidas de posição — **quartis** e **percentis** — permitem entender como os valores se distribuem ao longo do conjunto de dados, identificando padrões, concentrações e outliers nas variáveis analisadas.

Para isso, serão gerados **boxplots** das principais métricas ofensivas e defensivas, interpretando cada visualização no contexto da hipótese do projeto: o fator casa influencia o desempenho das equipes?

## Importação das bibliotecas

Nesta etapa, são importadas as bibliotecas necessárias para a análise:
- **Pandas**: manipulação e análise dos dados;
- **NumPy**: cálculo de percentis;
- **Matplotlib / Seaborn**: criação dos gráficos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

## Carregamento dos dados

Serão utilizados dois arquivos: o dataset bruto `games.csv`, que contém variáveis detalhadas de desempenho (porcentagem de arremessos, assistências e rebotes), e o arquivo tratado `clean_games.csv`, que já passou pelo pré-processamento realizado nas etapas anteriores do projeto.

In [ ]:
games_raw = pd.read_csv('../data/raw/games.csv')
games     = pd.read_csv('../data/processed/clean_games.csv')

print('Registros no dataset bruto:', games_raw.shape[0])
print('Registros no dataset tratado:', games.shape[0])

## Variáveis selecionadas para análise

Para as medidas de posição, foram selecionadas as principais variáveis de desempenho disponíveis no dataset:

| Variável | Descrição |
|---|---|
| `PTS_home` / `PTS_away` | Pontos marcados pelo time da casa / visitante |
| `FG_PCT_home` / `FG_PCT_away` | Aproveitamento geral de arremessos (%) |
| `FT_PCT_home` / `FT_PCT_away` | Aproveitamento de lances livres (%) |
| `FG3_PCT_home` / `FG3_PCT_away` | Aproveitamento de arremessos de 3 pontos (%) |
| `AST_home` / `AST_away` | Assistências |
| `REB_home` / `REB_away` | Rebotes |

In [ ]:
cols_home = ['PTS_home', 'FG_PCT_home', 'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home']
cols_away = ['PTS_away', 'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away', 'REB_away']

games_raw[cols_home + cols_away].describe().round(3)

## Quartis e Percentis da Pontuação

Os **quartis** dividem os dados ordenados em quatro partes iguais:
- **Q1 (25%)**: 25% dos jogos tiveram pontuação abaixo deste valor;
- **Q2 (50%)** — mediana: metade dos jogos ficou acima e metade abaixo;
- **Q3 (75%)**: 75% dos jogos ficaram abaixo deste valor.

Os **percentis** generalizam esse conceito. Aqui também serão calculados o **P10** e o **P90**, que delimitam os 10% mais baixos e os 10% mais altos de cada distribuição.

In [ ]:
percentis = [10, 25, 50, 75, 90]

resultados = {}
for col in ['PTS_home', 'PTS_away']:
    resultados[col] = np.percentile(games_raw[col].dropna(), percentis)

df_percentis = pd.DataFrame(resultados, index=[f'P{p}' for p in percentis])
df_percentis.index.name = 'Percentil'
df_percentis.round(2)

### Interpretação — Pontuação

A tabela acima mostra os percentis de pontuação para times da casa (`PTS_home`) e visitantes (`PTS_away`).

- **P10**: os 10% de jogos com menor pontuação da casa ficaram abaixo de ~87 pontos, enquanto para os visitantes esse limiar é ainda menor (~84 pontos), evidenciando que jogos de baixíssima pontuação são mais frequentes entre visitantes.
- **P50 (mediana)**: a mediana dos mandantes (103 pts) supera a dos visitantes (100 pts) — diferença consistente com o fator casa.
- **P90**: nos jogos de alta pontuação os mandantes chegam a ~121 pontos contra ~118 dos visitantes, reforçando o padrão de vantagem em casa em todos os níveis da distribuição.

## Quartis e Percentis — Todas as variáveis de desempenho

A seguir, o cálculo é expandido para todas as métricas selecionadas, permitindo uma visão completa das medidas de posição do dataset.

In [ ]:
tabela_home = pd.DataFrame(
    {col: np.percentile(games_raw[col].dropna(), percentis) for col in cols_home},
    index=[f'P{p}' for p in percentis]
).T
tabela_home.index.name = 'Variável (Casa)'

tabela_away = pd.DataFrame(
    {col: np.percentile(games_raw[col].dropna(), percentis) for col in cols_away},
    index=[f'P{p}' for p in percentis]
).T
tabela_away.index.name = 'Variável (Visitante)'

print('=== Times da Casa ===')
display(tabela_home.round(3))

print('\n=== Times Visitantes ===')
display(tabela_away.round(3))

## Boxplot — Distribuição da Pontuação (Casa vs. Fora)

O **boxplot** é uma representação gráfica direta das medidas de posição:
- A **caixa** vai do Q1 ao Q3 (intervalo interquartílico — IQR);
- A **linha interna** representa a mediana (Q2);
- Os **bigodes** se estendem até 1,5× o IQR;
- Os pontos além dos bigodes são **outliers**.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

data_plot = pd.DataFrame({
    'Casa': games_raw['PTS_home'].dropna().values,
    'Visitante': games_raw['PTS_away'].dropna().values
})

bp = ax.boxplot(
    [data_plot['Casa'], data_plot['Visitante']],
    labels=['Casa', 'Visitante'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2.5),
    whiskerprops=dict(color='gray'),
    capprops=dict(color='gray'),
    flierprops=dict(marker='o', color='gray', alpha=0.3, markersize=3)
)

cores = ['#4C72B0', '#DD8452']
for patch, cor in zip(bp['boxes'], cores):
    patch.set_facecolor(cor)
    patch.set_alpha(0.75)

for i, col in enumerate(['PTS_home', 'PTS_away'], 1):
    q1, med, q3 = np.percentile(games_raw[col].dropna(), [25, 50, 75])
    ax.text(i, med + 1.5, f'Mediana: {med:.0f}', ha='center', va='bottom', fontsize=9, color='navy', fontweight='bold')
    ax.text(i + 0.27, q1, f'Q1={q1:.0f}', ha='left', va='center', fontsize=8, color='dimgray')
    ax.text(i + 0.27, q3, f'Q3={q3:.0f}', ha='left', va='center', fontsize=8, color='dimgray')

ax.set_title('Distribuição da Pontuação: Times da Casa vs. Visitantes', fontsize=13, fontweight='bold')
ax.set_ylabel('Pontos por Jogo')
ax.set_xlabel('Localização')
plt.tight_layout()
plt.savefig('../visualizations/boxplot_pontuacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em: visualizations/boxplot_pontuacao.png')

### Interpretação — Boxplot de Pontuação

O boxplot evidencia que:
- A **mediana dos times da casa** é superior à dos visitantes, confirmando a vantagem no mando de quadra.
- O **IQR** (tamanho da caixa) é semelhante entre os dois grupos, indicando dispersão parecida.
- A presença de **outliers** em pontuações muito baixas ocorre em ambos os grupos, mas com maior frequência entre os visitantes — o que pode indicar que derrotas expressivas acontecem mais frequentemente longe de casa.
- Os valores do Q1 ao Q3 dos times da casa se posicionam consistentemente acima dos visitantes, reforçando a hipótese do projeto.

## Boxplot — Comparação por variável de desempenho

A seguir, são apresentados boxplots lado a lado para cada uma das métricas selecionadas, permitindo comparar visualmente a distribuição das equipes em casa e fora em diferentes aspectos do jogo.

In [ ]:
variaveis = [
    ('PTS_home',    'PTS_away',    'Pontuação'),
    ('FG_PCT_home', 'FG_PCT_away', 'Aproveit. Arremessos (FG%)'),
    ('FT_PCT_home', 'FT_PCT_away', 'Aproveit. Lances Livres (FT%)'),
    ('FG3_PCT_home','FG3_PCT_away','Aproveit. 3 Pontos (3P%)'),
    ('AST_home',    'AST_away',    'Assistências'),
    ('REB_home',    'REB_away',    'Rebotes'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Medidas de Posição: Boxplots Casa vs. Visitante', fontsize=15, fontweight='bold', y=1.01)

colores = ['#4C72B0', '#DD8452']

for ax, (col_h, col_a, titulo) in zip(axes.flat, variaveis):
    data_h = games_raw[col_h].dropna().values
    data_a = games_raw[col_a].dropna().values

    bp = ax.boxplot(
        [data_h, data_a],
        labels=['Casa', 'Visitante'],
        patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(color='gray'),
        capprops=dict(color='gray'),
        flierprops=dict(marker='o', alpha=0.2, markersize=2)
    )

    for patch, cor in zip(bp['boxes'], colores):
        patch.set_facecolor(cor)
        patch.set_alpha(0.75)
    for flier, cor in zip(bp['fliers'], colores):
        flier.set(markerfacecolor=cor)

    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_ylabel('Valor')

plt.tight_layout()
plt.savefig('../visualizations/boxplots_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em: visualizations/boxplots_metricas.png')

### Interpretação — Boxplots por Métrica

A análise dos seis boxplots revela padrões importantes sobre o fator casa:

- **Pontuação (PTS)**: os mandantes apresentam mediana e IQR superiores, confirmando maior eficiência ofensiva jogando em casa.
- **Aproveitamento de Arremessos (FG%)**: os times da casa tendem a ter distribuição ligeiramente mais alta, com Q3 superior ao dos visitantes — sugerindo maior precisão quando atuam diante de sua torcida.
- **Lances Livres (FT%)**: a distribuição é mais semelhante entre os grupos, com IQR parecido, indicando que os lances livres sofrem menos influência do fator casa.
- **Arremessos de 3 Pontos (3P%)**: alta variabilidade em ambos os grupos (bigodes longos e outliers), refletindo a natureza imprevisível do arremesso de longa distância.
- **Assistências (AST)**: a distribuição dos mandantes tende a se concentrar em valores um pouco mais altos, sugerindo melhor circulação de bola em casa.
- **Rebotes (REB)**: a diferença é pequena, mas os times da casa mostram Q3 ligeiramente superior, indicando maior agressividade nas disputas pelo aro dentro de casa.

## Classificação por Faixas de Desempenho (Percentis)

Nesta etapa, os jogos são classificados em **quintis** de pontuação do time da casa — ou seja, cinco faixas de igual tamanho baseadas nos percentis P20, P40, P60 e P80.

Essa abordagem permite verificar se a taxa de vitórias do time da casa é influenciada pela faixa de pontuação atingida.

In [ ]:
limites = np.percentile(games_raw['PTS_home'].dropna(), [20, 40, 60, 80])
rotulos = ['Muito Baixa\n(P0–P20)', 'Baixa\n(P20–P40)', 'Média\n(P40–P60)', 'Alta\n(P60–P80)', 'Muito Alta\n(P80–P100)']

games_raw_copy = games_raw.copy()
games_raw_copy['faixa_pts_casa'] = pd.cut(
    games_raw_copy['PTS_home'],
    bins=[-np.inf] + list(limites) + [np.inf],
    labels=rotulos
)

taxa = games_raw_copy.groupby('faixa_pts_casa', observed=False)['HOME_TEAM_WINS'].mean().round(3) * 100
print('Taxa de vitória da casa por faixa de pontuação (%):')
print(taxa.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(
    taxa.index, taxa.values,
    color=plt.cm.Blues(np.linspace(0.35, 0.85, len(taxa))),
    edgecolor='white', linewidth=0.8
)

for bar, val in zip(bars, taxa.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(y=50, color='red', linestyle='--', linewidth=1, label='50% (Equilíbrio)')
ax.set_title('Taxa de Vitória do Time da Casa por Faixa de Pontuação (Quintis)', fontsize=12, fontweight='bold')
ax.set_ylabel('Taxa de Vitória (%)')
ax.set_xlabel('Faixa de Pontuação (Quintil)')
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/vitoria_por_faixa_percentil.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em: visualizations/vitoria_por_faixa_percentil.png')

### Interpretação — Taxa de Vitória por Quintil

O gráfico acima demonstra de forma clara que **quanto maior a pontuação do time da casa, maior a taxa de vitória**. A análise por percentis adiciona precisão a essa relação:

- Nas faixas de pontuação **muito baixa e baixa** (abaixo do P40), os times da casa vencem menos de 50% dos jogos — o que significa que, quando não pontuam bem, perdem o benefício do mando de quadra.
- Na faixa **média** (P40–P60), a taxa de vitória já supera 50%, confirmando que a mediana de pontuação já é suficiente para garantir vantagem competitiva.
- Nas faixas **alta e muito alta** (acima do P60), as taxas de vitória aumentam expressivamente, chegando a superar 80% nos jogos de maior pontuação.

Essa análise reforça que o **fator casa se manifesta especialmente quando os mandantes atingem pontuações acima da mediana** da distribuição.

## Resumo das Medidas de Posição

A tabela abaixo consolida os quartis calculados para todas as variáveis analisadas, facilitando a comparação entre times da casa e visitantes.

In [ ]:
resumo_rows = []
for col_h, col_a, nome in [
        ('PTS_home',    'PTS_away',    'Pontuação'),
        ('FG_PCT_home', 'FG_PCT_away', 'FG%'),
        ('FT_PCT_home', 'FT_PCT_away', 'FT%'),
        ('FG3_PCT_home','FG3_PCT_away','3P%'),
        ('AST_home',    'AST_away',    'Assistências'),
        ('REB_home',    'REB_away',    'Rebotes'),
]:
    q1h, q2h, q3h = np.percentile(games_raw[col_h].dropna(), [25, 50, 75])
    q1a, q2a, q3a = np.percentile(games_raw[col_a].dropna(), [25, 50, 75])
    resumo_rows.append({
        'Variável':          nome,
        'Q1 Casa':           round(q1h, 3),
        'Mediana Casa':      round(q2h, 3),
        'Q3 Casa':           round(q3h, 3),
        'Q1 Visitante':      round(q1a, 3),
        'Mediana Visitante': round(q2a, 3),
        'Q3 Visitante':      round(q3a, 3),
    })

df_resumo = pd.DataFrame(resumo_rows).set_index('Variável')
df_resumo

## Conclusões

A análise das medidas de posição — quartis e percentis — trouxe evidências quantitativas relevantes para o problema investigado pelo grupo:

1. **Os times da casa apresentam Q1, mediana e Q3 superiores** em praticamente todas as métricas ofensivas analisadas (pontuação, aproveitamento de arremessos e assistências), confirmando a vantagem do mando de quadra em toda a distribuição.

2. **Nos rebotes e lances livres**, a diferença entre os grupos é menor, sugerindo que esses aspectos do jogo sofrem menos influência do fator casa.

3. **A análise por quintis de pontuação** revelou que a vantagem de jogar em casa só se concretiza de forma expressiva quando os times conseguem pontuações acima de sua mediana histórica — abaixo disso, o fator casa não garante vitória.

4. Os **boxplots** mostraram que as distribuições são simétricas e sem assimetrias graves, o que valida o uso de medidas como média e desvio padrão nas demais etapas do projeto.

Essas conclusões se conectam diretamente com as demais análises do grupo, especialmente as etapas de medidas de centralização e dispersão, e sustentam a hipótese central de que **o fator casa é uma vantagem real e mensurável no basquete da NBA**.